# Inference Notebook — AG News DistilBERT

This notebook loads the fine-tuned DistilBERT model from Hugging Face Hub and performs inference on sample data.

**Model**: [`Recurrent/ag-news-distilbert`](https://huggingface.co/Recurrent/ag-news-distilbert)  
**Task**: 4-class text classification (World, Sports, Business, Sci/Tech)

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers torch pandas datasets

## Step 2: Load Model and Tokenizer from Hugging Face Hub

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import pandas as pd

# Model repository on Hugging Face
HF_REPO_ID = "Recurrent/ag-news-distilbert"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(HF_REPO_ID)
model = AutoModelForSequenceClassification.from_pretrained(HF_REPO_ID)

print(f"Model loaded from: https://huggingface.co/{HF_REPO_ID}")
print(f"Number of labels: {model.config.num_labels}")
print(f"Labels: {model.config.id2label}")

## Step 3: Create Inference Pipeline

In [ ]:
# Create a text classification pipeline
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    top_k=None  # Return scores for all classes
)

print("Inference pipeline ready!")

## Step 4: Load Sample Test Data

In [ ]:
import json, requests
from datasets import load_dataset

# Load test data from Hugging Face Hub
HF_DATASET_REPO = "Recurrent/prepared_data_mlops2"
dataset = load_dataset(HF_DATASET_REPO, data_dir="prepared_data")
test_df = dataset["test"].to_pandas()

# Load id2label from GitHub
id2label_url = "https://raw.githubusercontent.com/riteshmaury-iitj/group13-assignment-mlops/ritesh_dev/id2label.json"
id2label = {int(k): v for k, v in requests.get(id2label_url).json().items()}

# Take a sample of 3 examples per class
sample_df = test_df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(n=min(3, len(x)), random_state=42)
).reset_index(drop=True)

print(f"Total test samples: {len(test_df)}")
print(f"Selected sample size: {len(sample_df)}")
print(f"Labels: {id2label}")
sample_df.head(12)

## Step 5: Run Inference on Sample Data

In [ ]:
# Run inference
texts = sample_df["text"].tolist()
predictions = classifier(texts, truncation=True, max_length=128)

# Extract predicted labels and confidence scores
predicted_labels = []
confidences = []

for pred in predictions:
    # Get the top prediction (highest score)
    top_pred = max(pred, key=lambda x: x["score"])
    predicted_labels.append(top_pred["label"])
    confidences.append(top_pred["score"])

# Build results dataframe
results_df = pd.DataFrame({
    "text": [t[:100] + "..." if len(t) > 100 else t for t in texts],
    "true_label": sample_df["label"].map(id2label).values,
    "predicted_label": predicted_labels,
    "confidence": confidences
})

results_df

## Step 6: Evaluate Inference Accuracy

In [ ]:
# Calculate accuracy on sample
correct = (results_df["true_label"] == results_df["predicted_label"]).sum()
total = len(results_df)
accuracy = correct / total

print(f"Sample Inference Results:")
print(f"  Correct: {correct}/{total}")
print(f"  Accuracy: {accuracy:.2%}")
print(f"  Average Confidence: {results_df['confidence'].mean():.4f}")

## Step 7: Inference on Custom Text

In [ ]:
import re
import string

def clean_text(text):
    """Clean and normalise text (same logic as prepare_data.ipynb)."""
    if pd.isna(text) or text is None:
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove HTML entities and tags
    text = re.sub(r'&lt;.*?&gt;', '', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&quot;', '"', text)
    text = re.sub(r'&#?\w+;', '', text)

    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)

    # Remove backslashes
    text = text.replace('\\', ' ')

    # Replace #36; encoding with $
    text = re.sub(r'#36;', '$', text)

    # Strip punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Custom text inputs
custom_texts = [
    "The stock market surged today as investors reacted to positive earnings reports from major tech companies.",
    "NASA successfully launched a new satellite to study climate change patterns in the Arctic region.",
    "Manchester United defeated Liverpool 3-1 in an exciting Premier League match at Old Trafford.",
    "The United Nations Security Council held an emergency meeting to discuss the ongoing conflict in the region."
]

# Apply the same data preparation logic used during training
custom_texts_cleaned = [clean_text(text) for text in custom_texts]

print("Custom Text Inference Results:")
print("=" * 80)

custom_predictions = classifier(custom_texts_cleaned, truncation=True, max_length=128)

for text, cleaned, pred in zip(custom_texts, custom_texts_cleaned, custom_predictions):
    top_pred = max(pred, key=lambda x: x["score"])
    print(f"\nOriginal: {text[:80]}...")
    print(f"Cleaned:  {cleaned[:80]}...")
    print(f"  Predicted: {top_pred['label']} (confidence: {top_pred['score']:.4f})")
    print(f"  All scores: {', '.join(f'{p[\"label\"]}: {p[\"score\"]:.3f}' for p in sorted(pred, key=lambda x: -x['score']))}")